#   Forecasting de Demanda Eléctrica en España
## Análisis de Series Temporales con Machine Learning y Variables Meteorológicas

---

###   Descripción del Proyecto

Este proyecto tiene como objetivo **predecir la demanda eléctrica en España** utilizando modelos estadísticos (SARIMA) y modelos de Machine Learning, incorporando **variables meteorológicas y energéticas** como factores explicativos clave.

**Dataset:** [Energy Consumption, Generation, Prices and Weather - Spain (Kaggle)](https://www.kaggle.com/datasets/nicholasjhana/energy-consumption-generation-prices-and-weather)

**Contexto:**
Según un paper de 2019, el forecasting en mercados energéticos es una de las áreas de mayor impacto del Machine/Deep Learning para la transición hacia una infraestructura eléctrica basada en renovables.

**Contenido del dataset:**
- **4 años de datos horarios** (2015-2018) 
- Consumo eléctrico real y forecast del TSO
- **Generación por fuente**: solar, eólica, gas, carbón, nuclear, hidráulica
- **Precios spot** y day-ahead
- **Datos meteorológicos** de las 5 ciudades más grandes de España:
  - Madrid, Barcelona, Valencia, Sevilla, Bilbao
  - Variables: temperatura, presión, humedad, velocidad del viento

**Fuentes:**
- ENTSOE (European Network of Transmission System Operators)
- Red Eléctrica de España (REE)
- Open Weather API

---

###   Objetivos Principales

1. **Limpieza y preparación** de datos con tratamiento correcto de valores nulos
2. **Análisis meteorológico**: Demostrar la relación entre clima y demanda eléctrica
3. **Feature Engineering avanzado**: Variables temporales, lags, rolling stats, y **weather features**
4. **Modelado estadístico**: SARIMA con análisis de tendencia, estacionalidad y residuo
5. **Machine Learning**: Random Forest y Gradient Boosting
6. **Análisis técnico**: Medias móviles, Golden Cross y volatilidad
7. **Comparación de modelos** y análisis de feature importance

**Hipótesis clave:**
> Las **variables meteorológicas** (especialmente temperatura) son factores explicativos fundamentales para la demanda eléctrica, junto con la información temporal (lags y estacionalidad).

---

###   Equipo de Trabajo y Responsabilidades

####   **Persona 1: Preparación de Datos y EDA**

**Tareas:**
-   Carga y exploración inicial del dataset
-   Conversión a series temporales (índice datetime)
-   Análisis de valores nulos y estrategia de imputación
-   Análisis exploratorio de demanda eléctrica
-   Análisis meteorológico: correlación clima-demanda
-   Visualizaciones de patrones diarios/semanales/estacionales

**Entregables:**
- Dataset limpio y correctamente indexado
- Informe de calidad de datos
- Gráficos exploratorios de demanda y temperatura

---

####   **Persona 2: Análisis Temporal y Comparación de Modelos**

**Tareas:**

-   **Lags**: 1h, 24h, 168h (capturar autocorrelación)
-   **Descomposición temporal**: tendencia, estacionalidad (period=24), residuo
-   **Test de estacionariedad** (ADF)
-	**ACF**
-   **Modelo SARIMA**: determinación de órdenes (p,d,q)(P,D,Q,s)
-   **Media móvil y Golden Cross** para precios eléctricos
-   **Análisis de volatilidad**

**Entregables:**
- Modelo SARIMA con parámetros justificados
- Análisis de cruces de medias móviles en precios
- Demostración de que temperatura y lags son cruciales

---

####   **Persona 3: Machine Learning**

**Tareas:**
-   **Modelos ML**: Random Forest y Gradient Boosting
-   **Feature importance**: identificar variables más predictivas
-   **Comparación final de modelos**: SARIMA vs RF vs GB
-   Visualizaciones de predicciones vs valores reales

**Entregables:**
- Modelos ML con hiperparámetros optimizados
- Análisis de feature importance
- Tabla comparativa de métricas (RMSE, MAE, MAPE)
- Conclusiones sobre mejor modelo y variables importantes

---

###   Preguntas de Investigación

1. **¿Qué ciudades/mediciones meteorológicas influyen más en la demanda eléctrica?**
2. **¿Podemos predecir 24 horas adelante mejor que el TSO?**
3. **¿Cuál es la relación entre temperatura y demanda?** (hipótesis: forma de U)
4. **¿Qué features son más importantes**: lags, temperatura, hora del día, generación?
5. **¿Qué modelo funciona mejor**: SARIMA, RF o Gradient Boosting?

---

---

##   1. Importación de Librerías

Importamos todas las librerías necesarias para el análisis completo.

In [22]:
import pandas as pd
import numpy as np

---

##   2. Carga y Exploración Inicial del Dataset



In [23]:
# 1. Cargar dataset de energía
df_energy = pd.read_csv('./data/energy_dataset.csv')
print(f"Energy dataset: {df_energy.shape[0]:,} filas × {df_energy.shape[1]} columnas")

# 2. Cargar dataset de clima 
df_weather = pd.read_csv('./data/weather_features.csv')
print(f"Weather dataset: {df_weather.shape[0]:,} filas × {df_weather.shape[1]} columnas")

print("\n" + "=" * 80)
print("ESTRUCTURA DE LOS DATASETS:")
print("=" * 80)

print("\nEnergy Dataset:")
print(df_energy.info())

print("\nWeather Dataset:")
print(df_weather.info())


Energy dataset: 35,064 filas × 29 columnas
Weather dataset: 178,396 filas × 17 columnas

ESTRUCTURA DE LOS DATASETS:

Energy Dataset:
<class 'pandas.DataFrame'>
RangeIndex: 35064 entries, 0 to 35063
Data columns (total 29 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   time                                         35064 non-null  str    
 1   generation biomass                           35045 non-null  float64
 2   generation fossil brown coal/lignite         35046 non-null  float64
 3   generation fossil coal-derived gas           35046 non-null  float64
 4   generation fossil gas                        35046 non-null  float64
 5   generation fossil hard coal                  35046 non-null  float64
 6   generation fossil oil                        35045 non-null  float64
 7   generation fossil oil shale                  35046 non-null  float64
 8   generation fossil peat 

---

## 2b. Separación de Columnas de Forecast (Comparación con TSO)

### Objetivo Principal

El propósito de este notebook es **desarrollar nuestro propio pronóstico de demanda eléctrica** utilizando modelos estadísticos (SARIMA) y Machine Learning. Las columnas de forecast del TSO las utilizaremos como **referencia de comparación** para comparar nuestra previsión con la oficial de red electrica.

### Posible Data Leakage

Las columnas de **forecast del TSO** (pronóstico oficial del operador del sistema) representan una **potencial fuente de data leakage**:

- Si usamos estas columnas para entrenar nuestro modelo, estaríamos utilizando información que **no estaría disponible en un escenario real de predicción en tiempo real**.
- Esto podría inflar artificialmente la precisión de nuestro modelo.

### Estrategia: Separar para Comparación Limpia

**Acción:** Extraer columnas de forecast a dataset separado **ANTES del merge**, para:
1. ✓ Entrenar nuestros modelos **sin estas columnas** (pronóstico limpio)
2. ✓ **Comparar nuestras predicciones** con el pronóstico oficial del TSO
3. ✓ Responder: **¿Predecimos mejor o peor que el TSO?**

In [24]:
# Identificar y separar columnas de forecast (TSO) antes del merge
forecast_cols = [col for col in df_energy.columns if 'forecast' in col.lower() or 'ahead' in col.lower()]

print(f"Columnas de forecast encontradas: {len(forecast_cols)}")
print("Columnas identificadas:")
for col in forecast_cols:
    print(f"  - {col}")

# 1. Crear dataset separado con forecast (para comparación posterior)
df_forecast_tso = df_energy[['time'] + forecast_cols].copy()

print(f"\nDataset de forecast creado: {df_forecast_tso.shape}")
print(f"Rango: {df_forecast_tso['time'].min()} → {df_forecast_tso['time'].max()}")

# 2. Eliminar columnas de forecast del dataset principal (evitar data leakage)
df_energy = df_energy.drop(columns=forecast_cols)

Columnas de forecast encontradas: 5
Columnas identificadas:
  - forecast solar day ahead
  - forecast wind offshore eday ahead
  - forecast wind onshore day ahead
  - total load forecast
  - price day ahead

Dataset de forecast creado: (35064, 6)
Rango: 2015-01-01 00:00:00+01:00 → 2018-12-31 23:00:00+01:00


In [25]:
# Convertir 'time' a datetime en ambos (con utc=True para manejar el cambio de horario verano/invierno)
df_energy['time'] = pd.to_datetime(df_energy['time'], utc=True)
df_weather['time'] = pd.to_datetime(df_weather['dt_iso'], utc=True)
df_weather.drop(columns='dt_iso', inplace=True)

# Merge por columna 'time' (inner join para mantener solo fechas comunes)
df = pd.merge(df_energy, df_weather, on='time', how='inner')

print(f"Dataset combinado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Rango de fechas: {df['time'].min()} → {df['time'].max()}")
print(f"Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

df.info()

Dataset combinado: 178,396 filas × 40 columnas
Rango de fechas: 2014-12-31 23:00:00+00:00 → 2018-12-31 22:00:00+00:00
Memoria utilizada: 87.03 MB
<class 'pandas.DataFrame'>
RangeIndex: 178396 entries, 0 to 178395
Data columns (total 40 columns):
 #   Column                                       Non-Null Count   Dtype              
---  ------                                       --------------   -----              
 0   time                                         178396 non-null  datetime64[us, UTC]
 1   generation biomass                           178301 non-null  float64            
 2   generation fossil brown coal/lignite         178306 non-null  float64            
 3   generation fossil coal-derived gas           178306 non-null  float64            
 4   generation fossil gas                        178306 non-null  float64            
 5   generation fossil hard coal                  178306 non-null  float64            
 6   generation fossil oil                        178301 n

---

## 2c. Eliminación de Columnas Innecesarias

Durante la exploración inicial identificamos columnas que **no aportan valor predictivo** o presentan **problemas de calidad/coherencia**. Procedemos a eliminarlas:

### Columnas a Eliminar:

1. **`total load actual`**  
   - Calculada por el autor del dataset de Kaggle
   - No cuadra matemáticamente con las demás columnas de load
   - Puede causar inconsistencias en el análisis

2. **`temp_min` y `temp_max`**  
   - En datos **horarios** no tiene sentido tener temperatura mínima/máxima
   - Estos valores tienen sentido solo en **agregaciones diarias**
   - Mejor usar `temp` (temperatura actual por hora)

3. **`weather_id`**  
   - Identificador interno de la API de OpenWeather
   - Redundante con `weather_main` y `weather_description`
   - No aporta información interpretable

4. **`weather_icon`**  
   - Código de iconos para interfaces gráficas (`01d`, `02n`, etc.)
   - Información redundante con otras columnas meteorológicas
   - No útil para modelado predictivo

In [26]:
# Columnas a eliminar
cols_to_drop = [
    'total load actual',    # Columna calculada por autor, inconsistente
    'temp_min',             # No tiene sentido en datos horarios
    'temp_max',             # No tiene sentido en datos horarios
    'weather_id',           # ID interno de API, no útil
    'weather_icon'          # Código de iconos, redundante
]

# Verificar cuáles existen realmente en el dataset
cols_existing = [col for col in cols_to_drop if col in df.columns]
cols_missing = [col for col in cols_to_drop if col not in df.columns]

print(f"Dataset antes: {df.shape}")
print(f"\nColumnas a eliminar ({len(cols_existing)}):")
for col in cols_existing:
    print(f"   - {col}")

if cols_missing:
    print(f"\n⚠️  Columnas no encontradas ({len(cols_missing)}):")
    for col in cols_missing:
        print(f"   - {col}")

# Eliminar columnas existentes
df = df.drop(columns=cols_existing)

print(f"\nDataset después: {df.shape}")
print(f"Columnas eliminadas: {len(cols_existing)}")

Dataset antes: (178396, 40)

Columnas a eliminar (5):
   - total load actual
   - temp_min
   - temp_max
   - weather_id
   - weather_icon

Dataset después: (178396, 35)
Columnas eliminadas: 5


---

##   3. Conversión a Series Temporales



**Fundamental:** Convertir la columna `time` a formato datetime y establecerla como índice para trabajar correctamente con series temporales.

In [27]:
# Establecer como índice
df =df.set_index('time')

# Ordenar cronológicamente
df = df.sort_index()

# Verificar frecuencia
print(f"\nPrimera fecha: {df.index.min()}")
print(f"Última fecha: {df.index.max()}")
print(f"📊 Total observaciones: {len(df):,}")
df.head(3)


Primera fecha: 2014-12-31 23:00:00+00:00
Última fecha: 2018-12-31 22:00:00+00:00
📊 Total observaciones: 178,396


,generation biomass,generation fossil brown coal/lignite,generation fossil coal-derived gas,generation fossil gas,generation fossil hard coal,generation fossil oil,generation fossil oil shale,generation fossil peat,generation geothermal,generation hydro pumped storage aggregated,...,pressure,humidity,wind_speed,wind_deg,rain_1h,rain_3h,snow_3h,clouds_all,weather_main,weather_description
time,,,,,,,,,,,,,,,,,,,,,
2014-12-31 23:00:00+00:00,447.0,329.0,0.0,4844.0,4821.0,162.0,0.0,0.0,0.0,NaN,...,1001,77,1,62,0.0,0.0,0.0,0,clear,sky is clear
2014-12-31 23:00:00+00:00,447.0,329.0,0.0,4844.0,4821.0,162.0,0.0,0.0,0.0,NaN,...,971,63,1,309,0.0,0.0,0.0,0,clear,sky is clear
2014-12-31 23:00:00+00:00,447.0,329.0,0.0,4844.0,4821.0,162.0,0.0,0.0,0.0,NaN,...,1036,97,0,226,0.0,0.0,0.0,0,clear,sky is clear


---

## 4. Análisis de Valores Nulos

Identificamos y cuantificamos los valores nulos en el dataset. Es importante distinguir entre **valores 0.0** (generación nula registrada) y **valores NaN** (datos no reportados).

### Columnas con 0.0 (Generación Nula Real)

Estas columnas tienen **toda su serie en 0.0** porque España **no utiliza estas fuentes energéticas**:

1. **`generation fossil oil shale`** (Esquistos bituminosos)  
   España no usa esquistos para generación eléctrica comercial.

2. **`generation fossil peat turb`** (Turba)  
   No hay centrales de turba operativas en España.

3. **`generation geothermal`** (Geotermia)  
   A excepción de pequeños proyectos en Canarias, España no produce electricidad geotérmica a gran escala.

4. **`generation marine`** (Energía marina/undimotriz)  
   Tecnología experimental. No aporta energía significativa a la red nacional.

5. **`generation wind offshore`** (Eólica marina)  
   Durante 2015-2018 (periodo del dataset), la eólica marina era **inexistente** en España.

### Columnas con NaN (Datos No Reportados)

Valores **NaN** significan que el dato **no existe o no se reportó**:

1. **`generation hydro pumped storage aggregated`**  
   Las empresas reportan por separado consumo (bombeo hacia arriba) y generación (turbinado). La columna "agregada" queda vacía porque españa identifica la energía generada en otro concepto (generation hydro water reservoir) y da error la agregación.


### Estrategia de Tratamiento

- **Columnas con todo 0.0**: Eliminar (no aportan variabilidad ni información predictiva)
- **Columnas con NaN puntuales**: Aplicar interpolación temporal o forward/backward fill
- **Columnas con todo NaN**: Eliminar (sin información útil)

In [28]:
# Columnas a eliminar
cols_to_drop = [
    'generation fossil oil shale',    
    'generation fossil coal-derived gas',             
    'generation fossil peat',            
    'generation geothermal',          
    'generation hydro pumped storage aggregated',
    'generation marine',
    'generation wind offshore'

]

# Verificar cuáles existen realmente en el dataset
cols_existing = [col for col in cols_to_drop if col in df.columns]
cols_missing = [col for col in cols_to_drop if col not in df.columns]

print(f"Dataset antes: {df.shape}")
print(f"\nColumnas a eliminar ({len(cols_existing)}):")
for col in cols_existing:
    print(f"   - {col}")

if cols_missing:
    print(f"\n⚠️  Columnas no encontradas ({len(cols_missing)}):")
    for col in cols_missing:
        print(f"   - {col}")

# Eliminar columnas existentes
df = df.drop(columns=cols_existing)

print(f"\nDataset después: {df.shape}")
print(f"Columnas eliminadas: {len(cols_existing)}")

Dataset antes: (178396, 34)

Columnas a eliminar (7):
   - generation fossil oil shale
   - generation fossil coal-derived gas
   - generation fossil peat
   - generation geothermal
   - generation hydro pumped storage aggregated
   - generation marine
   - generation wind offshore

Dataset después: (178396, 27)
Columnas eliminadas: 7


---

## 5. Tratamiento de Valores Faltantes

### Estrategia de Imputación

Basándonos en el análisis anterior, aplicamos diferentes técnicas según el tipo de problema:

1. **Eliminar columnas con todo 0.0**  
   No aportan variabilidad ni información (generación inexistente en España)

2. **Eliminar columnas con >90% de NaN**  
   Información insuficiente para imputación confiable

3. **Interpolación temporal** para valores nulos puntuales  
   Método apropiado para series temporales porque asume cambios graduales entre valores consecutivos

4. **Forward/Backward Fill** como respaldo  
   Para NaN al inicio/final de la serie donde la interpolación no funciona

In [29]:
df_clean = df.copy()

# Método 1: Interpolación lineal (mantiene tendencia temporal)
columnas_numericas = df_clean.select_dtypes(include=[np.number]).columns

for col in columnas_numericas:
    if df_clean[col].isna().sum() > 0:
        antes = df_clean[col].isna().sum()
        df_clean[col] = df_clean[col].interpolate(method='linear', limit_direction='both')
        despues = df_clean[col].isna().sum()
        print(f"{col}: {antes} → {despues} nulos")


generation biomass: 95 → 0 nulos
generation fossil brown coal/lignite: 90 → 0 nulos


generation fossil gas: 90 → 0 nulos
generation fossil hard coal: 90 → 0 nulos
generation fossil oil: 95 → 0 nulos
generation hydro pumped storage consumption: 95 → 0 nulos
generation hydro run-of-river and poundage: 95 → 0 nulos
generation hydro water reservoir: 90 → 0 nulos
generation nuclear: 85 → 0 nulos
generation other: 90 → 0 nulos
generation other renewable: 90 → 0 nulos
generation solar: 90 → 0 nulos
generation waste: 95 → 0 nulos
generation wind onshore: 90 → 0 nulos


---

##  6. Análisis Meteorológico



Las **variables meteorológicas son fundamentales** para explicar y predecir la demanda eléctrica. Vamos a demostrar la relación entre **temperatura y demanda**.

---

##   7. Feature Engineering Completo



El **feature engineering es CRÍTICO** para el éxito de los modelos ML. Crearemos:
1. Features temporales (hora, día, mes)
2. Lags (autocorrelación)
3. Rolling statistics (tendenciaslocales)
4. **Weather features** (lags y rolling de temperatura)  
5. Variables energéticas (renovables vs fósiles)


##   7. Feature Engineering - Parte A: Variables Temporales



La creación de features temporales es **fundamental** para capturar patrones diarios, semanales y estacionales en la demanda eléctrica.

**Features cíclicas:** Las transformaciones sin/cos permiten que los modelos ML entiendan que la hora 23 está cerca de la hora 0.

---

##   8. Feature Engineering - Parte B: Lags (Autocorrelación)



Los **lags** capturan la autocorrelación temporal: el valor actual depende de valores pasados.

**Lags clave:**
- **lag_1**: Valor hace 1 hora
- **lag_24**: Valor hace 24 horas (ayer misma hora)
- **lag_168**: Valor hace 168 horas (semana pasada misma hora)

---

##   9. Feature Engineering - Parte C: Rolling Statistics



Las **rolling statistics** (estadísticas móviles) capturan tendencias y volatilidad reciente.

**Ventanas importantes:**
- **24 horas**: Captura tendencia/volatilidad del último día
- **168 horas**: Captura tendencia/volatilidad de la última semana

---

##   10. Feature Engineering - Parte D: **Weather Features** 




Las **weather features son FUNDAMENTALES** para predecir demanda eléctrica. La temperatura afecta directamente el consumo (calefacción en invierno, aire acondicionado en verano).

**Features meteorológicas clave:**
- Lags de temperatura (temp_lag24, temp_lag168)
- Rolling statistics de temperatura
- Interacciones con hora del día

---

##   11. Feature Engineering - Parte E: Variables Energéticas



Creamos agregaciones de las fuentes de generación eléctrica para simplificar el análisis.

---

##  12. Descomposición Temporal (Tendencia, Estacionalidad, Residuo)



La descomposición temporal permite separar la serie en:
- **Tendencia**: Patrón de largo plazo
- **Estacionalidad**: Patrón repetitivo (diario/semanal)
- **Residuo**: Variación aleatoria

---

##  13. Test de Estacionariedad (ADF Test)



El **Test de Dickey-Fuller Aumentado (ADF)** determina si la serie es estacionaria.

**Estacionaria:** Media y varianza constantes en el tiempo → se puede modelar con ARIMA/SARIMA directamente

**No estacionaria:** Requiere **diferenciación** para estabilizar la media.

---

##   14. Análisis Técnico: Media Móvil y Golden Cross



Aplicamos técnicas de **análisis técnico** (usadas en finanzas) al **precio eléctrico**.

**Medias Móviles:**
- **MA_24**: Media móvil de 24 horas (corto plazo)
- **MA_168**: Media móvil de 168 horas (largo plazo / 1 semana)

**Golden Cross:** Cuando MA corto cruza por encima de MA largo → señal alcista (precios subiendo)

---

##   15. Análisis de Volatilidad



La **volatilidad** mide la variabilidad de los precios/demanda en el tiempo.

**Volatilidad = Rolling Standard Deviation**

Alta volatilidad → Mayor riesgo/incertidumbre en el mercado

---

##  16. Modelo SARIMA (Seasonal ARIMA)



**SARIMA(p,d,q)(P,D,Q,s)** modela series temporales con estacionalidad.

**Parámetros:**
- **(p,d,q)**: Componente no estacional (AR, diferenciación, MA)
- **(P,D,Q,s)**: Componente estacional con periodo s=24 (horas)

**Nota:** Entrenar SARIMA con todos los datos toma mucho tiempo. Usaremos una muestra para demostración.

---

##  17. Preparación para Machine Learning



Preparamos el dataset con features engineered para modelos de Machine Learning.

**Importante:** División temporal (NO train_test_split aleatorio) para respetar el orden cronológico.

---

##  18. Modelo Random Forest



**Random Forest** es un ensemble de árboles de decisión que funciona muy bien para series temporales con features engineered.

**Ventaja:** Proporciona **feature importance** que nos permite identificar las variables más predictivas.

---

##   19. Modelo Gradient Boosting (LightGBM/XGBoost)



**Gradient Boosting** suele ser el modelo MÁS POTENTE para problemas de regresión y forecasting.

Usaremos **LightGBM** o **XGBoost** (dependiendo de disponibilidad).

---

##  20. Comparación Final de Modelos



Comparamos todos los modelos entrenados para determinar cuál es el mejor.

---

##  21. Visualizaciones Finales: Predicciones vs Valores Reales



Visualización de las predicciones de todos los modelos comparadas con los valores reales.

---

##   22. Conclusiones y Hallazgos Clave

###   Responsable: Todos
